# STEP 04 — Feature Selection (변수 선택)

**목적:** Random Forest Feature Importance를 이용해 One-Hot 이후 **전체 / 상위 5 / 10 / 15개** 특성을 비교한다. Logistic Regression과 Random Forest의 **5-Fold 교차검증 결과**를 기준으로 선택 효과를 판단한다.

> 이 파일은 **STEP 04만** 수행하며 독립적으로 실행된다. STEP 05 파일이나 이전 노트북의 변수를 불러오지 않는다.
>
> **검증 원칙:** 각 fold의 훈련 부분에서 `결측치 처리 → 인코딩 → 스케일링 → 변수 선택`을 학습한다. 본 노트북의 특성 선택 비교에는 홀드아웃 테스트 데이터를 사용하지 않는다. STEP03에서 동일 분할을 탐색한 이력은 별도의 평가 한계로 구분한다.
>
> STEP 03의 `Median + One-Hot + StandardScaler`와 파생 변수 3종을 고정한다.

## 1. 데이터 로드 및 분할

In [ ]:
# 패키지 설치: %pip install pandas numpy matplotlib scikit-learn joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

SEED = 42
CSV_PATH = Path('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f'{CSV_PATH} 파일을 찾지 못했습니다. 노트북이 들어있는 프로젝트 폴더에서 실행하고 '
        '필수 입력 파일: data/WA_Fn-UseC_-Telco-Customer-Churn.csv.'
    )

df = pd.read_csv(CSV_PATH)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
X = df.drop(columns=['customerID','Churn'])
y = df['Churn'].map({'No':0, 'Yes':1})
if y.isna().any():
    raise ValueError('Churn에 허용 범주 No/Yes 외의 값이 포함되어 있다.')
y = y.astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
OUT_DIR = Path('results')
OUT_DIR.mkdir(exist_ok=True)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print('전체:', df.shape, '| 학습:', X_train.shape, '| 테스트:', X_test.shape)
print('학습 이탈률:', f'{y_train.mean():.2%}', '| 테스트 이탈률:', f'{y_test.mean():.2%}')
print('TotalCharges 결측:', df['TotalCharges'].isna().sum())

## 2. 파생 변수 + 전처리 + 선택 Pipeline
- 파생 변수: `TenureGroup`, `ServiceCount`, `AvgChargeRatio`
- 수치형 결측: 중앙값 / 수치형 스케일링: StandardScaler
- 범주형 결측: 최빈값 / 범주형 인코딩: One-Hot
- 선택: Random Forest 기반 Top-K; `all`이면 선택 생략

**중요:** Top 5/10/15는 원본 컬럼 수가 아니라 **One-Hot 변환 후의 특성 수**를 의미한다.

In [ ]:
SERVICE_COLUMNS = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                   'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
NUMERIC_COLS = ['SeniorCitizen','tenure','MonthlyCharges','TotalCharges',
                'ServiceCount','AvgChargeRatio']
CATEGORY_COLS = ['gender','Partner','Dependents','PhoneService','MultipleLines',
                 'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
                 'TechSupport','StreamingTV','StreamingMovies','Contract',
                 'PaperlessBilling','PaymentMethod','TenureGroup']

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """각 행의 값으로만 파생 변수를 만들며, 목표값 Churn은 사용하지 않음."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        frame = X.copy()
        frame['TenureGroup'] = pd.cut(
            frame['tenure'], [-1,12,24,48,np.inf],
            labels=['0-12m','13-24m','25-48m','49m+']
        ).astype('object')
        frame['ServiceCount'] = sum(
            frame[col].eq('Yes').astype(int) for col in SERVICE_COLUMNS
        )
        denom = frame['MonthlyCharges'] * frame['tenure']
        frame['AvgChargeRatio'] = (
            frame['TotalCharges'] / denom.replace(0, np.nan)
        ).replace([np.inf,-np.inf], np.nan)
        return frame

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:  # scikit-learn 1.2 이전 호환
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), NUMERIC_COLS),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', make_ohe())
        ]), CATEGORY_COLS)
    ], sparse_threshold=0)

class RFTopKSelector(BaseEstimator, TransformerMixin):
    """훈련 fold에서만 RF 중요도를 학습하는 Top-K 특성 선택기."""
    def __init__(self, k='all', n_estimators=40, random_state=42):
        self.k = k
        self.n_estimators = n_estimators
        self.random_state = random_state

    def fit(self, X, y):
        if self.k == 'all':
            self.indices_ = np.arange(X.shape[1])
        else:
            forest = RandomForestClassifier(
                n_estimators=self.n_estimators, random_state=self.random_state,
                class_weight='balanced', n_jobs=1
            )
            forest.fit(X, y)
            self.importances_ = forest.feature_importances_
            k = min(int(self.k), X.shape[1])
            self.indices_ = np.argsort(self.importances_)[::-1][:k]
        return self

    def transform(self, X):
        return X[:, self.indices_]

def make_pipeline(model, k='all'):
    return Pipeline([
        ('features', FeatureEngineer()),
        ('preprocess', make_preprocessor()),
        ('select', RFTopKSelector(k=k, random_state=SEED)),
        ('model', model)
    ])

print('Pipeline 단계: features -> preprocess -> select -> model')

## 3. 훈련 데이터 기반 변수 중요도 (Top 15)

이 시각화는 **전체 훈련 데이터만으로 계산한 설명용 중요도**다. 아래 순위를 교차검증에 그대로 재사용하지 않는다. 교차검증에서는 fold마다 선택기가 별도로 적합된다.

In [ ]:
importance_pipeline = Pipeline([
    ('features', FeatureEngineer()),
    ('preprocess', make_preprocessor()),
    ('forest', RandomForestClassifier(
        n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=1
    ))
])
importance_pipeline.fit(X_train, y_train)
feature_names = importance_pipeline.named_steps['preprocess'].get_feature_names_out()
importance_values = importance_pipeline.named_steps['forest'].feature_importances_
importance_df = (pd.DataFrame({'변수':feature_names, '중요도':importance_values})
                 .sort_values('중요도', ascending=False).reset_index(drop=True))
display(importance_df.head(15).round(5))

fig, ax = plt.subplots(figsize=(10,6))
importance_df.head(15).iloc[::-1].plot.barh(
    x='변수', y='중요도', ax=ax, legend=False
)
ax.set(title='STEP 04: Random Forest Feature Importance (Train Only)',
       xlabel='Importance', ylabel='')
plt.tight_layout()
plt.savefig(OUT_DIR/'step04_feature_importance.png', dpi=170, bbox_inches='tight')
plt.show()
importance_df.to_csv(OUT_DIR/'step04_feature_importance.csv', index=False, encoding='utf-8-sig')

## 4. 변수 선택 전후 성능 비교 (5-Fold CV)

두 모델 모두 **기본 하이퍼파라미터를 고정**하고 전체/Top 5/10/15 특성 조합만 비교한다. 이 단계에서는 GridSearchCV를 하지 않으며 하이퍼파라미터 최적화는 **STEP 05**에서 수행한다.

In [ ]:
from sklearn.model_selection import cross_validate

MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=3000, random_state=SEED),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=1
    )
}
K_CHOICES = ['all', 5, 10, 15]
SCORING = {'F1':'f1', 'ROC-AUC':'roc_auc', 'Accuracy':'accuracy',
           'Precision':'precision', 'Recall':'recall'}

cv_rows = []
for name, model in MODELS.items():
    for k in K_CHOICES:
        print(f'교차검증 중: {name} / {k}개')
        pipe = make_pipeline(model, k=k)
        score = cross_validate(
            pipe, X_train, y_train, cv=CV, scoring=SCORING,
            n_jobs=1, error_score='raise'
        )
        cv_rows.append({
            '모델':name, '선택 변수 수':str(k),
            **{f'CV {metric}':np.mean(score[f'test_{metric}'])
               for metric in SCORING},
            'CV F1 표준편차':np.std(score['test_F1'])
        })

cv_df = pd.DataFrame(cv_rows)
display(cv_df.round(4))
cv_df.to_csv(OUT_DIR/'step04_cv_comparison.csv', index=False, encoding='utf-8-sig')

## 5. 그래프: 선택 변수 수에 따른 CV F1-score

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
for name, group in cv_df.groupby('모델', sort=False):
    group = group.set_index('선택 변수 수').reindex(['all','5','10','15'])
    ax.plot(group.index, group['CV F1'], marker='o', label=name)
ax.set(title='STEP 04: 5-Fold CV F1 by Feature Count',
       xlabel='Selected features (all = no selection)', ylabel='Mean CV F1')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR/'step04_cv_f1_comparison.png', dpi=170)
plt.show()

## 6. 교차검증 기준 변수 수 결정 및 저장

각 모델에서 가장 높은 **훈련 CV F1-score**를 기록한 변수 수를 찾는다. 이 결과는 실험 보고서에 사용하며, STEP 05에서는 하이퍼파라미터까지 포함하여 최적 조합을 다시 평가한다.

In [ ]:
best_by_cv = (cv_df.sort_values('CV F1', ascending=False)
              .drop_duplicates('모델')
              [['모델','선택 변수 수','CV F1','CV ROC-AUC']]
              .sort_values('모델').reset_index(drop=True))
display(best_by_cv.round(4))
best_by_cv.to_csv(OUT_DIR/'step04_best_by_cv.csv', index=False, encoding='utf-8-sig')
print('STEP 04 완료! 저장 폴더:', OUT_DIR.resolve())
print('홀드아웃 테스트 데이터는 사용하지 않았습니다.')

## 분석 결과의 해석 범위

- 중요도 그래프는 전체 학습 데이터에 적합한 RF의 설명용 결과다.
- CV는 fold 내부에서 전처리와 특성 선택을 다시 적합한다.
- Top 5/10/15는 인코딩 이후 특성 수를 의미한다.
- 불순도 기반 중요도는 연속형 변수에 유리할 수 있으며 인과관계를 나타내지 않는다.
- STEP05는 훈련 CV로 변수 수와 하이퍼파라미터를 함께 선택한다.

**저장 파일:** `step04_feature_importance.png`, `step04_feature_importance.csv`, `step04_cv_f1_comparison.png`, `step04_cv_comparison.csv`, `step04_best_by_cv.csv`